Classificação percentílica regional e recortes municipais

In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append("/code/scripts")

from analysis_config import (
    ANALYSIS,
    structural_maps,
    seasonal_maps
)
from classification_utils import (
    percentile_breaks,
    classify_raster,
    breaks_table
)
from pilot_utils import select_municipality, clip_raster

In [ ]:
pilots = [
    {
        "pilot": "fundao",
        "area": "centro",
        "name": "Fundão",
        "source": Path(
            "/code/data/raw/limites/caop/Continente_CAOP2025.gpkg"
        ),
        "layer": "cont_municipios",
        "field": "municipio",
        "expected_area_km2": None
    },
    {
        "pilot": "badajoz",
        "area": "extremadura",
        "name": "Badajoz",
        "source": Path(
            "/code/pen/paper/data/lineas_limite/SHP_ETRS89/"
            "recintos_municipales_inspire_peninbal_etrs89"
        ),
        "layer": None,
        "field": "NAMEUNIT",
        "expected_area_km2": (1300, 1600)
    }
]

N_BINS = 100000
YEAR = 2025

In [ ]:
all_breaks = []
all_inventory = []

for config in pilots:
    pilot = config["pilot"]
    area = config["area"]

    pilot_dir = ANALYSIS / "02_classes" / pilot
    regional_dir = ANALYSIS / "02_classes" / area / "regional"
    local_dir = pilot_dir / "rasters"

    regional_dir.mkdir(parents=True, exist_ok=True)
    local_dir.mkdir(parents=True, exist_ok=True)

    municipality = pilot_dir / f"municipality_{pilot}.gpkg"

    select_municipality(
        source=config["source"],
        layer=config["layer"],
        field=config["field"],
        value=config["name"],
        output=municipality,
        expected_area_km2=config["expected_area_km2"]
    )

    records = structural_maps(area) + seasonal_maps(area, YEAR)

    for record in records:
        source = Path(record["raster"])

        if not source.exists():
            raise FileNotFoundError(
                f"Mapa em falta para {record['map_id']}: {source}"
            )

        regional_class = regional_dir / f"{record['map_id']}_class.tif"
        local_class = local_dir / f"{record['map_id']}_class_{pilot}.tif"

        print("\nMapa:", record["map_id"])
        breaks = percentile_breaks(
            source,
            n_bins=N_BINS
        )

        counts = classify_raster(
            source,
            breaks,
            regional_class
        )

        clip_raster(
            regional_class,
            municipality,
            local_class
        )

        table = breaks_table(
            record["map_id"],
            source,
            breaks,
            counts
        )
        table["area"] = area
        table["pilot"] = pilot
        all_breaks.append(table)

        all_inventory.append({
            **record,
            "pilot": pilot,
            "regional_class_raster": str(regional_class),
            "local_class_raster": str(local_class),
            "municipality": str(municipality)
        })

    inventory = pd.DataFrame(
        row for row in all_inventory
        if row["pilot"] == pilot
    )

    inventory.to_csv(
        pilot_dir / f"class_inventory_{pilot}.csv",
        index=False
    )

In [ ]:
breaks_df = pd.concat(all_breaks, ignore_index=True)
inventory_df = pd.DataFrame(all_inventory)

output = ANALYSIS / "02_classes" / "class_breaks_and_inventory.xlsx"

with pd.ExcelWriter(output) as writer:
    breaks_df.to_excel(
        writer,
        sheet_name="Class_breaks",
        index=False
    )
    inventory_df.to_excel(
        writer,
        sheet_name="Map_inventory",
        index=False
    )

print("Resultados guardados em:", output)
inventory_df[["pilot", "map_id", "local_class_raster"]]